# NYU Depth V2: RGB, depth, class labels, instances

Reads [NYU Depth V2](https://cs.nyu.edu/~fergus/datasets/nyu_depth_v2.html)'s labeled subset (1449 RGB-D pairs with dense
per-pixel class and instance labels) directly from its own `nyu_depth_v2_labeled.mat` with `h5py` -- the file is
MATLAB v7.3 (HDF5), so `scipy.io.loadmat` can't read it. Needs `INSTALL.MD`'s NYU Depth V2 section run first, i.e.
`$HABITAT_DATA/nyu_depth_v2/nyu_depth_v2_labeled.mat`.

In [ ]:
import os
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
os.environ.setdefault("BASE_DIR", str(REPO_ROOT))
NYU_ROOT = Path("data") / "nyu_depth_v2"
dataset = h5py.File(NYU_ROOT / "nyu_depth_v2_labeled.mat", "r")
list(dataset.keys())

## Array layout

MATLAB stores arrays column-major, so `h5py` reads each array back transposed relative to its MATLAB shape:
`images` comes back `(N, 3, W, H)`, `depths`/`labels`/`instances` come back `(N, W, H)`. Transpose each frame to
`(H, W, [3])` before displaying. `labels` is 0 for unlabeled pixels, 1-894 for a NYU-specific per-pixel class id
(mapped to names via `names`/`namesToIds`); `instances` is 0 for unlabeled, else an instance index that restarts at 1 for
every frame *and every class*: an object is a (`labels`, `instances`) pair, so `instances == 1` covers the first object
of every class in the frame at once.

In [ ]:
from habitat_embodied_sl.nyu_depth_v2 import class_names, frame_instances

NAMES = ["<unlabeled>"] + class_names(dataset)


def read_frame(dataset: h5py.File, index: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """(rgb, depth, labels, instances) for frame index, each transposed back to (H, W, ...)."""
    rgb = dataset["images"][index].transpose(2, 1, 0)
    depth = dataset["depths"][index].T
    labels = dataset["labels"][index].T
    instances = dataset["instances"][index].T
    return rgb, depth, labels, instances

## Drawing helpers

In [ ]:
_PALETTE = np.array(
    [(0, 0, 0), (230, 25, 75), (60, 180, 75), (255, 225, 25), (0, 130, 200), (245, 130, 48),
     (145, 30, 180), (70, 240, 240), (240, 50, 230), (210, 245, 60), (250, 190, 212)],
    dtype=np.uint8,
)


def colorize_ids(ids: np.ndarray) -> np.ndarray:
    """ids (e.g. labels or instances) as an RGB image, 0 mapped to black, every other id cycled through the palette."""
    palette_index = np.where(ids == 0, 0, (ids - 1) % (len(_PALETTE) - 1) + 1)
    return _PALETTE[palette_index]


def colorize_depth(depth: np.ndarray) -> np.ndarray:
    normalized = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
    return (plt.cm.viridis(normalized)[..., :3] * 255).astype(np.uint8)

## A few examples

In [ ]:
def show_example(index: int) -> None:
    rgb, depth, labels, instances = read_frame(dataset, index)
    present_classes = ", ".join(NAMES[i] for i in np.unique(labels) if i > 0)

    panels = [
        ("RGB", rgb),
        ("depth", colorize_depth(depth)),
        ("class labels", colorize_ids(labels)),
        ("instances", colorize_ids(instances)),
    ]
    _, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, (title, panel) in zip(axes, panels):
        ax.imshow(panel)
        ax.set_title(title)
        ax.axis("off")
    plt.suptitle(f"frame {index}: {present_classes}")
    plt.tight_layout()
    plt.show()

In [ ]:
for index in [0, 500, 1000]:
    show_example(index)

## Register as a detectron2 dataset

Builds a `DatasetCatalog`/`MetadataCatalog` entry over every frame that has at least one labeled instance: one
instance-segmentation annotation per (`labels`, `instances`) pair, found by `habitat_embodied_sl/nyu_depth_v2.py`'s
`frame_instances`, so the instances here are the ones its test set (`datasets/nyu_depth_v2/test`) holds, wall, floor,
ceiling and unknown included. Masks are RLE-encoded with `common/utils/coco_utils.py`'s `encoded_masks`.
`file_name` is a synthetic per-frame locator, not a real path -- frames live inside the single `.mat` file, so
`DatasetCatalog.get` works but detectron2's default image-loading `DatasetMapper` won't resolve it.

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.structures import BoxMode
from pycocotools import mask as mask_util

from common.utils.coco_utils import encoded_masks

CLASS_IDS = {label: label - 1 for label in range(1, len(NAMES))}


def instance_annotations(labels: np.ndarray, instances: np.ndarray) -> list[dict]:
    """One annotation per (class, instance) pair of the frame."""
    ground_truth = frame_instances(labels, instances, CLASS_IDS)
    return [
        {"bbox": mask_util.toBbox(rle).tolist(), "bbox_mode": BoxMode.XYWH_ABS, "segmentation": rle, "category_id": class_id}
        for rle, class_id in zip(encoded_masks(ground_truth.gt_masks), ground_truth.gt_classes.tolist())
    ]


def nyu_dataset_dicts() -> list[dict]:
    dataset_dicts = []
    for index in range(dataset["images"].shape[0]):
        _, _, labels, instances = read_frame(dataset, index)
        annotations = instance_annotations(labels.astype(np.int32), instances.astype(np.int32))
        if not annotations:
            continue
        dataset_dicts.append({
            "file_name": f"<nyu_depth_v2_labeled.mat>#images[{index}]",
            "image_id": index,
            "height": labels.shape[0],
            "width": labels.shape[1],
            "annotations": annotations,
        })
    return dataset_dicts


_nyu_dataset_dicts = nyu_dataset_dicts()
if "nyu_depth_v2" not in DatasetCatalog:
    DatasetCatalog.register("nyu_depth_v2", lambda: _nyu_dataset_dicts)
    MetadataCatalog.get("nyu_depth_v2").set(thing_classes=NAMES[1:])



In [ ]:
print(len(DatasetCatalog.get("nyu_depth_v2")), len(NAMES[1:]))
print(sorted(NAMES[1:])[:100])

## Merge semantic synonyms (pre-count)

String-level normalization only (lowercase, flatten `_`/`-`/whitespace, strip a naive trailing plural) -- catches
spelling variants like `book`/`books`, not different-looking synonyms. Applied before counting, so the histogram
below is grouped, not raw.

In [ ]:
from collections import defaultdict


def canonicalize(name: str) -> str:
    """Flattened, lowercased name with a naive trailing plural stripped -- merges 'book'/'books' without touching
    words that already end in 's' (e.g. 'glass')."""
    normalized = " ".join(name.lower().replace("_", " ").replace("-", " ").split())
    return normalized[:-1] if normalized.endswith("s") and not normalized.endswith("ss") else normalized


def merge_synonym_counts(raw_counts: dict[str, int]) -> tuple[dict[str, int], list[list[str]]]:
    """Groups raw_counts' keys by canonicalize(), keeping the most frequent raw spelling per group as its
    representative and summing the group's counts under it. merged_groups lists only groups with more than
    one raw spelling, for auditing."""
    groups = defaultdict(list)
    for name in raw_counts:
        groups[canonicalize(name)].append(name)

    merged_counts, merged_groups = {}, []
    for group in groups.values():
        representative = max(group, key=lambda name: raw_counts[name])
        merged_counts[representative] = sum(raw_counts[name] for name in group)
        if len(group) > 1:
            merged_groups.append(sorted(group, key=lambda name: -raw_counts[name]))
    return merged_counts, merged_groups

## Class frequency

Exhaustive per-class instance count from the registered dataset (`_nyu_dataset_dicts`), log-scale x since NYU's
per-object vocabulary is extremely long-tailed. Only classes with at least one instance appear.

In [ ]:
from collections import Counter

nyu_classes = NAMES[1:]
instance_counts = Counter(ann["category_id"] for record in _nyu_dataset_dicts for ann in record["annotations"])
raw_counts = {nyu_classes[i]: n for i, n in instance_counts.items()}
merged_counts, merged_groups = merge_synonym_counts(raw_counts)
print(f"merged {sum(len(g) for g in merged_groups)} raw classnames into {len(merged_groups)} groups, e.g.:")
for group in sorted(merged_groups, key=lambda g: -sum(raw_counts[n] for n in g))[:15]:
    print(" ", " + ".join(group))

counts_sorted = sorted(merged_counts.items(), key=lambda x: -x[1])
names, counts = zip(*counts_sorted)

fig, ax = plt.subplots(figsize=(8, max(6, len(names) * 0.09)))
ax.barh(range(len(names)), counts, color="#2a78d6")
ax.set_yticks(range(len(names)))
ax.set_yticklabels(list(enumerate(names)), fontsize=6)
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlabel("number of instances (log scale)")
ax.set_title(f"instance count per class, synonyms merged ({len(names)} classes, {sum(counts)} instances)")
plt.tight_layout()
plt.show()

In [ ]:
counts_sorted

In [ ]:
frequent_classes_nyu = sorted([n for n, c in counts_sorted if c > 10])
print(len(frequent_classes_nyu))
print(frequent_classes_nyu)